In [4]:
# -*- coding: utf-8 -*-
"""Untitled2.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1uso1W7SvzFPJfMfLbgBdt-_s93yxKmks
"""

!pip install streamlit pyngrok google-genai requests

# Commented out IPython magic to ensure Python compatibility.
# %%writefile app.py
# import streamlit as st
# from agent import job_agent
#
# st.title("AI Job Search Assistant")
#
# query = st.text_input("Enter job role (example: python developer)")
#
# if st.button("Search Jobs"):
#
#     jobs = job_agent(query)
#
#     if not jobs:
#         st.write("No jobs found")
#
#     else:
#         for job in jobs:
#             st.subheader(job["title"])
#             st.write("Company:", job["company"])
#             st.write("Location:", job["location"])
#             st.write("Apply:", job["url"])
#             st.write("---")

!streamlit run app.py &>/dev/null &

!ngrok config add-authtoken 3AXWdLYFp21qNmp7Va6dBB4ZRoC_35rWntc9adaGaNCuBr9GZ

!streamlit run app.py &>/dev/null &

!killall ngrok
!killall streamlit

!streamlit run app.py &>/dev/null &

"""# Task
Terminate all existing `ngrok` processes and then connect a new `ngrok` tunnel to expose the Streamlit application running on port 8501.

## Terminate Ngrok Processes Forcefully

### Subtask:
Terminate all ngrok related processes to ensure no conflicts when starting a new tunnel.

**Reasoning**:
The subtask requires terminating all ngrok processes to prevent conflicts. The instruction explicitly states to use `!pkill -f ngrok` for this purpose.
"""

!pkill -f ngrok
print("Attempted to terminate all ngrok processes.")

"""## Connect Ngrok Tunnel

### Subtask:
Attempt to establish a new `ngrok` tunnel to expose the Streamlit application.

## Summary:

### Data Analysis Key Findings
- The command `pkill -f ngrok` was executed to forcefully terminate any running `ngrok` processes.
- The execution of the termination command was confirmed by the message: "Attempted to terminate all ngrok processes."
- No errors were reported during the termination process, indicating a successful attempt to clear existing `ngrok` instances.

### Insights or Next Steps
- Successfully terminating existing `ngrok` processes is a crucial prerequisite for establishing a new, clean `ngrok` tunnel without conflicts.

# Task
Connect a new ngrok tunnel to expose the Streamlit application.

## Terminate Ngrok Processes Forcefully

### Subtask:
Increase the delay after forcefully terminating `streamlit` and `ngrok` processes to allow the `ngrok` cloud service sufficient time to release the endpoint.

**Reasoning**:
The subtask requires increasing the delay after terminating Streamlit and ngrok processes. This code block combines all the necessary commands, including importing `time`, forcefully killing `streamlit` and `ngrok` processes, calling `ngrok.kill()`, and then introducing a 30-second delay.
"""

import time

!killall -9 streamlit
!pkill -f ngrok
from pyngrok import ngrok

ngrok.kill()
time.sleep(30) # Increased delay to ensure ngrok releases the endpoint
print("Streamlit and ngrok processes terminated, and a 30-second delay introduced.")

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
ngrok: no process found
Attempted to terminate all ngrok processes.
Streamlit and ngrok processes terminated, and a 30-second delay introduced.


In [5]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)


MessageError: Error: credential propagation was unsuccessful

In [ ]:
%%writefile app.py
import streamlit as st
import PyPDF2
import os
from google import genai
from agent import job_agent

# Initialize the Gemini client
# (Make sure os.environ["GEMINI_API_KEY"] is set in your notebook before running Streamlit)
client = genai.Client()

st.title("AI Job Search & Resume Matcher")

# --- 1. RESUME UPLOAD ---
st.header("1. Upload Profile")
uploaded_file = st.file_uploader("Upload your resume (PDF)", type=["pdf"])
resume_text = ""

if uploaded_file is not None:
    try:
        pdf_reader = PyPDF2.PdfReader(uploaded_file)
        for page in pdf_reader.pages:
            resume_text += page.extract_text() + "\n"
        st.success("Resume processed successfully!")
    except Exception as e:
        st.error(f"Error reading PDF: {e}")

# --- 2. PREFERENCES ---
st.header("2. Job Preferences")
preferences = st.text_input(
    "Any specific preferences?",
    placeholder="e.g., Remote, Entry-level, Cybersecurity, Node.js backend"
)

# --- 3. MATCHING LOGIC ---
st.header("3. Find Matches")
if st.button("Find Matched Jobs"):
    if not resume_text and not preferences:
        st.warning("Please upload a resume or enter some preferences first.")
    else:
        with st.spinner("Analyzing profile and finding matches..."):
            try:
                # Ask Gemini to extract core skills and blend them with preferences
                prompt = f"""
                Analyze this resume text and the user's preferences to create a highly targeted 3 to 5 word job search query.
                Resume snippet: {resume_text[:2500]}
                Preferences: {preferences}

                Return ONLY the raw search query string, nothing else. No formatting, no intro.
                """

                response = client.models.generate_content(
                    model='gemini-2.5-flash',
                    contents=prompt,
                )

                smart_query = response.text.strip()
                st.info(f"🧠 **AI-Optimized Search:** Searching for `{smart_query}`")

                # Feed the optimized query into your existing job agent
                jobs = job_agent(smart_query)

                if not jobs:
                    st.write("No matching jobs found. Try adjusting your preferences.")
                else:
                    st.subheader("Top Job Matches")
                    for job in jobs:
                        st.write(f"### {job.get('title', 'Unknown Title')}")
                        st.write("**Company:**", job.get("company", "Unknown"))
                        st.write("**Location:**", job.get("location", "Unknown"))
                        st.markdown(f"[Apply Here]({job.get('url', '#')})")
                        st.write("---")

            except Exception as e:
                st.error(f"An error occurred during matching: {e}")

In [ ]:
%%writefile app.py
import streamlit as st
import PyPDF2
import os
from google import genai
from agent import job_agent

# Initialize the Gemini client
client = genai.Client()

st.title("AI Job Search & Career Assistant")

# --- UI TABS ---
# Using tabs keeps the app clean as we add more features
tab1, tab2, tab3 = st.tabs(["📄 Resume & Match", "🏢 Company Research", "🔍 Raw Job Search"])

with tab1:
    st.header("Upload Profile")
    uploaded_file = st.file_uploader("Upload your resume (PDF)", type=["pdf"])
    resume_text = ""

    if uploaded_file is not None:
        try:
            pdf_reader = PyPDF2.PdfReader(uploaded_file)
            for page in pdf_reader.pages:
                resume_text += page.extract_text() + "\n"
            st.success("Resume processed successfully!")
        except Exception as e:
            st.error(f"Error reading PDF: {e}")

    st.header("Find Matched Jobs")
    preferences = st.text_input(
        "Any specific preferences?",
        placeholder="e.g., Remote, Entry-level, Cybersecurity, Node.js"
    )

    if st.button("Find Matched Jobs", key="match_btn"):
        if not resume_text and not preferences:
            st.warning("Please upload a resume or enter some preferences first.")
        else:
            with st.spinner("Analyzing profile and finding matches..."):
                try:
                    prompt = f"""
                    Analyze this resume text and the user's preferences to create a highly targeted 3 to 5 word job search query.
                    Resume snippet: {resume_text[:2500]}
                    Preferences: {preferences}
                    Return ONLY the raw search query string.
                    """
                    response = client.models.generate_content(
                        model='gemini-2.5-flash',
                        contents=prompt,
                    )

                    smart_query = response.text.strip()
                    st.info(f"🧠 **AI-Optimized Search:** Searching for `{smart_query}`")

                    jobs = job_agent(smart_query)

                    if not jobs:
                        st.write("No matching jobs found.")
                    else:
                        for job in jobs:
                            st.write(f"### {job.get('title', 'Unknown Title')}")
                            st.write("**Company:**", job.get("company", "Unknown"))
                            st.write("**Location:**", job.get("location", "Unknown"))
                            st.markdown(f"[Apply Here]({job.get('url', '#')})")
                            st.write("---")
                except Exception as e:
                    st.error(f"An error occurred: {e}")

with tab2:
    st.header("Company Insights")
    st.write("Before you apply, get a quick breakdown of the company.")

    company_name = st.text_input("Enter a company name", placeholder="e.g., Google, Stripe, or a local startup")

    if st.button("Research Company"):
        if not company_name:
            st.warning("Please enter a company name.")
        else:
            with st.spinner(f"Pulling insights for {company_name}..."):
                try:
                    research_prompt = f"""
                    Provide a concise, professional overview for a job seeker looking into the company: {company_name}.
                    Include the following sections using Markdown:
                    * **What they do:** (1-2 sentences)
                    * **Estimated Size & Industry:**
                    * **Known Tech Stack / Engineering Culture:** (If publicly known, otherwise skip)
                    * **Recent News or Focus Areas:** (What are they building right now?)

                    Keep it scannable and highly relevant for an engineer or tech professional.
                    """

                    research_response = client.models.generate_content(
                        model='gemini-2.5-flash',
                        contents=research_prompt,
                    )

                    st.markdown(research_response.text)
                except Exception as e:
                    st.error(f"Error researching company: {e}")

with tab3:
    st.header("Manual Job Search")
    manual_query = st.text_input("Search without resume", placeholder="e.g., python developer")
    if st.button("Search", key="manual_btn"):
        if manual_query:
            with st.spinner("Searching..."):
                jobs = job_agent(manual_query)
                if jobs:
                    for job in jobs:
                        st.write(f"**{job.get('title', 'Unknown')}** at {job.get('company', 'Unknown')}")
                        st.markdown(f"[Apply]({job.get('url', '#')})")
                else:
                    st.write("No jobs found.")

In [ ]:
import streamlit as st
# (Add your other imports here: PyPDF2, genai, etc.)

st.title("AI Job Search & Career Assistant")

# 1. DEFINE THE TABS FIRST
# This creates the variables tab1, tab2, and tab3
tab1, tab2, tab3 = st.tabs(["📄 Resume & Match", "🏢 Company Research", "🔍 Raw Job Search"])

# 2. NOW USE THE VARIABLES
with tab1:
    st.header("Upload Profile")
    uploaded_file = st.file_uploader("Upload your resume (PDF)", type=["pdf"])

    # ... rest of your logic for resume_text and preferences ...

    if st.button("Find Matched Jobs", key="match_btn"):
        # Ensure 'graph' is defined in a previous cell or block
        # result = graph.invoke({"query": preferences, "resume_text": resume_text})

        # Place your Gap Analysis logic INSIDE this button block
        st.divider()
        st.header("🎯 Skills Gap Analysis")
        # st.write(result.get("missing_skills"))

In [ ]:
# --- tab1: Resume & Match ---
with tab1:
    st.header("Upload Profile")
    # ... (your existing file_uploader and resume_text logic) ...

    if st.button("Find Matched Jobs", key="match_btn"):
        if not resume_text and not preferences:
            st.warning("Please upload a resume or enter some preferences first.")
        else:
            with st.spinner("Analyzing profile and finding matches..."):
                # 1. Invoke the graph (which now includes analyze_gap_node)
                # Ensure 'graph' is imported or defined in your notebook
                result = graph.invoke({
                    "query": preferences if preferences else "Software Engineer",
                    "resume_text": resume_text
                })

                # 2. Extract results from the state
                matched_jobs = result.get("matched_jobs", [])
                missing = result.get("missing_skills", [])
                recommendations = result.get("learning_path", [])

                if not matched_jobs:
                    st.write("No matching jobs found.")
                else:
                    # --- DISPLAY MATCHED JOBS ---
                    st.subheader("Top Job Matches")
                    for job in matched_jobs:
                        st.write(f"### {job.get('title', 'Unknown Title')}")
                        st.write(f"**Company:** {job.get('company', 'Unknown')}")
                        st.markdown(f"[Apply Here]({job.get('url', '#')})")
                        st.write("---")

                    # --- SKILLS GAP ANALYSIS (Nested inside the button logic) ---
                    st.divider()
                    st.header("🎯 Skills Gap Analysis")

                    if missing:
                        st.subheader("Skills to add for a 100% match:")
                        # Displaying missing skills as red tags
                        cols = st.columns(len(missing) if len(missing) > 0 else 1)
                        for idx, skill in enumerate(missing):
                            cols[idx % len(cols)].error(f"**{skill.upper()}**")

                        st.subheader("Personalized Learning Path")
                        for tip in recommendations:
                            st.info(f"💡 {tip}")
                    else:
                        st.success("🔥 Your profile is a perfect match for these roles!")

In [ ]:
%%writefile app.py
import streamlit as st
import PyPDF2
import os
from google import genai
from agent import job_agent

# Initialize the Gemini client
client = genai.Client()

st.title("AI Job Search & Career Assistant")

# 1. DEFINE THE TABS
tab1, tab2, tab3 = st.tabs(["📄 Resume & Match", "🏢 Company Research", "🔍 Raw Job Search"])

with tab1:
    st.header("Upload Profile")
    uploaded_file = st.file_uploader("Upload your resume (PDF)", type=["pdf"])

    # Initialize variables to prevent NameError
    resume_text = ""
    preferences = st.text_input(
        "Any specific preferences?",
        placeholder="e.g., Remote, Entry-level, Cybersecurity, Node.js"
    )

    if uploaded_file is not None:
        try:
            pdf_reader = PyPDF2.PdfReader(uploaded_file)
            for page in pdf_reader.pages:
                resume_text += page.extract_text() + "\n"
            st.success("Resume processed successfully!")
        except Exception as e:
            st.error(f"Error reading PDF: {e}")

    # --- MATCHING & ANALYSIS LOGIC ---
    if st.button("Find Matched Jobs", key="match_btn"):
        if not resume_text and not preferences:
            st.warning("Please upload a resume or enter some preferences first.")
        else:
            with st.spinner("Analyzing profile and generating tailored content..."):
                try:
                    # 1. Invoke the graph (assuming 'graph' is defined in your environment)
                    # We pass the full state to match your AgentState definition
                    result = graph.invoke({
                        "query": preferences if preferences else "Software Engineer",
                        "resume_text": resume_text,
                        "skills": [],
                        "jobs": [],
                        "matched_jobs": [],
                        "missing_skills": [],
                        "learning_path": [],
                        "tailored_resume": ""
                    })

                    # 2. Extract results from the state
                    matched_jobs = result.get("matched_jobs", [])
                    missing = result.get("missing_skills", [])
                    recommendations = result.get("learning_path", [])
                    tailored = result.get("tailored_resume", "")

                    if not matched_jobs:
                        st.write("No matching jobs found.")
                    else:
                        # --- SECTION: JOB MATCHES ---
                        st.subheader("Top Job Matches")
                        for job in matched_jobs:
                            st.write(f"### {job.get('title', 'Unknown Title')}")
                            st.write(f"**Company:** {job.get('company', 'Unknown')}")
                            st.markdown(f"[Apply Here]({job.get('url', '#')})")
                            st.write("---")

                        # --- SECTION: SKILLS GAP ANALYSIS ---
                        st.divider()
                        st.header("🎯 Skills Gap Analysis")
                        if missing:
                            st.subheader("Skills to add for a 100% match:")
                            cols = st.columns(len(missing))
                            for idx, skill in enumerate(missing):
                                cols[idx].error(f"**{skill.upper()}**")

                            st.subheader("Personalized Learning Path")
                            for tip in recommendations:
                                st.info(f"💡 {tip}")
                        else:
                            st.success("🔥 Your profile is a perfect match for these roles!")

                        # --- SECTION: TAILORED RESUME ---
                        if tailored:
                            st.divider()
                            st.header("📝 Tailored Resume Snippet")
                            st.info("AI has optimized your summary and projects for these roles.")
                            st.text_area("Optimized Summary & Projects", value=tailored, height=300)
                            st.download_button(
                                label="Download Tailored Content",
                                data=tailored,
                                file_name="tailored_resume.txt",
                                mime="text/plain"
                            )

                except Exception as e:
                    st.error(f"An error occurred: {e}")

# (Add Tab 2 and Tab 3 logic below as needed)

In [ ]:
# --- tab4: AI Mock Interview ---
with tab4:
    st.header("🤖 AI Mock Interviewer")
    st.info("The AI will ask you questions based on your profile. Type your answers below.")

    # Initialize chat history in Streamlit session state
    if "messages" not in st.session_state:
        st.session_state.messages = []

    # Display existing chat history
    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])

    # Chat Input
    if user_answer := st.chat_input("Your answer..."):
        # 1. Display user message
        st.session_state.messages.append({"role": "user", "content": user_answer})
        with st.chat_message("user"):
            st.markdown(user_answer)

        # 2. Get AI Response (Interviewer)
        with st.spinner("AI is thinking..."):
            # Prepare state for the graph/agent
            result = graph.invoke({
                "query": preferences if preferences else "Software Engineer",
                "resume_text": resume_text,
                "chat_history": st.session_state.messages
            })

            ai_response = result.get("current_question", "I'm ready for the next question.")

        # 3. Display AI message
        st.session_state.messages.append({"role": "assistant", "content": ai_response})
        with st.chat_message("assistant"):
            st.markdown(ai_response)